# NGB v4 small 4×4 tuned two-epoch baselines

Runs the distinct four-block/four-head small language-model architecture.

The test split is monitoring-only. Validation cross-entropy selects the best
checkpoint. Runtime data and results remain under explicit `/tmp` paths.


In [ ]:
CONFIG_PATH = "configs/v4_small_4x4.yaml"
OPTIMIZER = "all"
SEEDS = ""
DEVICE = "auto"
DATA_ROOT = "/tmp/rg-nanogpt-one-head/data"
NGB_STORAGE_ROOT = "/tmp/rg-ngb"
FORCE_DATA = False
OVERWRITE = False


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "baseline" / "ngb"]
NGB_ROOT_DIR = next(
    (path for path in candidates if (path / "configs" / "v4_one_head.yaml").is_file()),
    None,
)
if NGB_ROOT_DIR is None:
    raise FileNotFoundError("Run from baseline/ngb or the repository root")
RUNTIME_SRC = NGB_ROOT_DIR.parent / "nanogpt_one_head" / "src"
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

from rg_nanogpt_one_head import (
    SUPPORTED_OPTIMIZERS,
    canonical_seeds,
    load_config,
    prepare_fineweb_edu,
    run_all_replicates,
    run_optimizer_replicates,
    run_slug,
    run_status_table,
)


In [ ]:
config_path = (NGB_ROOT_DIR / CONFIG_PATH).resolve()
cfg = load_config(config_path)
selected_seeds = (
    tuple(int(value.strip()) for value in SEEDS.split(",") if value.strip())
    if SEEDS.strip()
    else canonical_seeds(cfg)
)
if not selected_seeds or len(set(selected_seeds)) != len(selected_seeds):
    raise ValueError("SEEDS must contain unique integers")
if OPTIMIZER != "all" and OPTIMIZER not in SUPPORTED_OPTIMIZERS:
    raise ValueError(f"OPTIMIZER must be all or one of {SUPPORTED_OPTIMIZERS}")

data_root = Path(DATA_ROOT)
results_root = Path(NGB_STORAGE_ROOT) / "results" / run_slug(cfg)
plots_root = Path(NGB_STORAGE_ROOT) / "plots" / run_slug(cfg)
for directory in (data_root, results_root, plots_root):
    directory.mkdir(parents=True, exist_ok=True)

print("config:", config_path)
print("data:", data_root)
print("results:", results_root)
print("seeds:", selected_seeds)
display(pd.DataFrame([cfg["model"]]))
display(pd.DataFrame.from_dict(cfg["optimizer_profiles"], orient="index"))

prepare_fineweb_edu(cfg, data_root, force=FORCE_DATA)
common = dict(
    cfg=cfg,
    config_path=config_path,
    seeds=selected_seeds,
    data_root=data_root,
    results_root=results_root,
    device=DEVICE,
    resume=not OVERWRITE,
    overwrite=OVERWRITE,
    progress=True,
)
if OPTIMIZER == "all":
    run_dirs = run_all_replicates(**common)
else:
    run_dirs = run_optimizer_replicates(
        optimizer_name=OPTIMIZER,
        prepare_data=False,
        **common,
    )
print("run directories:", len(run_dirs))
display(run_status_table(results_root, optimizers=SUPPORTED_OPTIMIZERS, seeds=selected_seeds))
